# Mamba — Selective SSM from scratch (NumPy)

**What this notebook is for**

This notebook implements the core complonent of Mamba's selective state space model (SSM) in pure NumPy, with no autograd and no training. 


The goal is not a trainable model, it's to prove the math from [Gu & Dao, 2023](https://arxiv.org/pdf/2312.00752) is understood and implemented correctly.


Trainable version of this using PyTorch lives in the sibling notebook `compare_mamba_with_transformers.ipynb`.

Complete implementation plan can be find in `Implementation_plan.md`

In [ ]:
import numpy as np

rng = np.random.default_rng(42)

## Data format

Shapes throughout (batch omitted — unit tests use a single sequence):

- **x** — input, `(L, D)`: L timesteps, D channels.
- **A** — transition matrix, `(D, N)`: diagonal SSM, so one value per channel-state pair, not a full N×N matrix.
- **delta (Δ)** — step size, `(L, D)`: per-timestep, per-channel.
- **B** — input matrix, `(L, N)`: shared across channels, varies per timestep — the selective part.
- **C** — output matrix, `(L, N)`: same shape and role as B.
- **A_bar (Ā)** — discretized transition, `(L, D, N)`: one per timestep since Δ varies.
- **B_bar (B̄)** — discretized input matrix, `(L, D, N)`.
- **h** — hidden state, `(D, N)` per step, full trace `(L, D, N)`.
- **y** — output, `(L, D)`: same shape as x.
- **W_B, W_C** — selection projection weights, `(D, N)`: map D channels to the N-dim state space, applied per timestep.
- **W_delta** — selection projection weight, `(D, 1)`: projects each channel to a scalar, then broadcasts over D.

Toy test shapes: L=4, D=3, N=2, fixed random seed for reproducibility.

## Core math

In [ ]:
def softplus(x):
    """Numerically stable softplus: log(1 + exp(x)).

    Used to keep Δ positive (required for the discretization to be a valid
    zero-order hold), matching the paper's use of softplus for Δ.

    Args:
        x: ndarray, any shape.

    Returns:
        ndarray, same shape as x.
    """
    raise NotImplementedError

In [ ]:
def discretize(delta, A, B):
    """Zero-order hold (ZOH) discretization of the continuous SSM.

    Ā = exp(Δ A)
    B̄ = A⁻¹ (Ā − I) B

    A is diagonal (stored as shape (D, N), one scalar per channel/state pair),
    so exp and the A⁻¹(Ā − I) term are elementwise rather than a matrix
    exponential / inverse.

    Args:
        delta: ndarray (L, D) — per-timestep, per-channel step size Δ.
        A: ndarray (D, N) — diagonal state transition (one value per
            channel/state pair).
        B: ndarray (L, N) — input matrix, varies per timestep.

    Returns:
        A_bar: ndarray (L, D, N) — discretized state transition Ā.
        B_bar: ndarray (L, D, N) — discretized input matrix B̄.
    """
    raise NotImplementedError

In [ ]:
def selective_scan(x, delta, A, B, C):
    """Sequential selective-scan recurrence.

    For t = 1..L:
        hₜ = Āₜ hₜ₋₁ + B̄ₜ xₜ
        yₜ = Cₜ hₜ

    h₀ is the zero vector. Runs as an explicit Python loop over timesteps
    (not vectorized/parallel-scan) — this is meant to mirror the paper's
    recurrence directly, not to be fast.

    Args:
        x: ndarray (L, D) — input sequence.
        delta: ndarray (L, D) — per-timestep, per-channel Δ.
        A: ndarray (D, N) — diagonal state transition.
        B: ndarray (L, N) — input matrix, varies per timestep.
        C: ndarray (L, N) — output matrix, varies per timestep.

    Returns:
        y: ndarray (L, D) — output sequence.
    """
    raise NotImplementedError

## Selection projections

These are what make the SSM *selective*: Δ, B, C are computed from the input
x at each timestep, rather than being fixed learned parameters.

In [ ]:
def s_B(x, W_B):
    """Selection projection for B: a linear map from input to state space.

    B = x @ W_B

    Args:
        x: ndarray (L, D) — input sequence.
        W_B: ndarray (D, N) — projection weights.

    Returns:
        ndarray (L, N) — per-timestep B.
    """
    raise NotImplementedError

In [ ]:
def s_C(x, W_C):
    """Selection projection for C: a linear map from input to state space.

    C = x @ W_C

    Args:
        x: ndarray (L, D) — input sequence.
        W_C: ndarray (D, N) — projection weights.

    Returns:
        ndarray (L, N) — per-timestep C.
    """
    raise NotImplementedError

In [ ]:
def s_delta(x, W_delta):
    """Selection projection for Δ: rank-1 linear map, then softplus, then
    broadcast over the channel dimension D.

    delta = softplus(x @ W_delta) broadcast to (L, D)

    Args:
        x: ndarray (L, D) — input sequence.
        W_delta: ndarray (D, 1) — projection weights (rank-1, per the paper).

    Returns:
        ndarray (L, D) — per-timestep, per-channel Δ, positive-valued.
    """
    raise NotImplementedError

## Full forward pass

In [ ]:
def selective_ssm_forward(x, A, W_B, W_C, W_delta):
    """Full selective-SSM forward pass: projections → discretize → scan.

    B = s_B(x, W_B); C = s_C(x, W_C); delta = s_delta(x, W_delta)
    A_bar, B_bar = discretize(delta, A, B)
    y = selective_scan(x, delta, A, B, C)

    Args:
        x: ndarray (L, D) — input sequence.
        A: ndarray (D, N) — diagonal state transition (learned, not
            input-dependent — only Δ, B, C are selective per the paper).
        W_B: ndarray (D, N) — selection projection weights for B.
        W_C: ndarray (D, N) — selection projection weights for C.
        W_delta: ndarray (D, 1) — selection projection weights for Δ.

    Returns:
        y: ndarray (L, D) — output sequence.
    """
    raise NotImplementedError

## Verification

Not part of the model itself — used to sanity-check `selective_ssm_forward`
against a numerical derivative, and (later) against the PyTorch reference.

In [ ]:
def numerical_gradient(f, x, eps=1e-5):
    """Central-difference numerical gradient of a scalar-valued function.

    For each entry x[i]: (f(x + eps*e_i) - f(x - eps*e_i)) / (2*eps)

    Args:
        f: callable, ndarray -> scalar. Should internally reduce
            selective_ssm_forward's output to a scalar (e.g. via .sum())
            so a gradient w.r.t. x is well-defined.
        x: ndarray, the point to differentiate around.
        eps: float, perturbation size.

    Returns:
        ndarray, same shape as x — the numerical gradient.
    """
    raise NotImplementedError

In [ ]:
def check_gradients(analytical_grad, numerical_grad, tol=1e-4):
    """Compare an analytical gradient to a numerical one.

    Args:
        analytical_grad: ndarray.
        numerical_grad: ndarray, same shape as analytical_grad.
        tol: float, max allowed elementwise absolute difference.

    Returns:
        bool — True if all elements agree within tol.
    """
    raise NotImplementedError

In [ ]:
def compare_to_pytorch(numpy_output, torch_output, tol=1e-5):
    """Compare this notebook's NumPy output to the PyTorch reference
    (from compare_mamba_with_transformers.ipynb), given identical inputs
    and weights.

    Args:
        numpy_output: ndarray (L, D).
        torch_output: array-like (L, D) — a torch.Tensor, e.g. detached and
            converted via .detach().numpy() before calling this function.
        tol: float, max allowed elementwise absolute difference.

    Returns:
        bool — True if all elements agree within tol.
    """
    raise NotImplementedError